# LeetCode 3411 — Maximum Subarray With Equal Products bằng Python

Link đề: https://leetcode.com/problems/maximum-subarray-with-equal-products/ · Độ khó: **Easy** · Chủ đề: Array, Math, Number Theory (gcd/lcm), Enumeration, Sliding Window

Skeleton LeetCode:

```python
class Solution:
    def maxLength(self, nums: List[int]) -> int:
        ...
```

Ghi chú: đề và ví dụ được dịch từ đề gốc (Weekly Contest 431); mọi lời giải trong notebook đã được chạy và kiểm chứng bằng code thật. Trong notebook ta viết `list[int]` thay cho `List[int]` để không cần `import`; khi nộp LeetCode cả hai đều chạy được.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: một mảng `arr` gọi là **product equivalent** nếu `prod(arr) == lcm(arr) * gcd(arr)`; tìm **độ dài lớn nhất** của một **mảng con** (đoạn liên tiếp) của `nums` có tính chất đó.
2. Nắm các công cụ Python: `math.gcd` (ước chung lớn nhất), `math.lcm` (bội chung nhỏ nhất), `math.prod` (tích), và `gcd`/`lcm` **nhiều đối số**; **mặt nạ bit** (`&`, `|`, `^`) để biểu diễn tập số nguyên tố; vòng lặp lồng nhau và **cửa sổ trượt (hai con trỏ)**.
3. Đi từ cách **tính trực tiếp tích/gcd/lcm của mọi mảng con** → **cập nhật dần khi kéo dài mảng con** → **kiểm tra "đôi một nguyên tố cùng nhau"** → **cửa sổ trượt với mặt nạ bit** → **cửa sổ trượt với phân tích thừa số nguyên tố tổng quát**.
4. Hiểu mấu chốt toán học: với hai số luôn có `gcd · lcm = a · b`, còn với **từ ba số trở lên**, `prod == lcm · gcd` **khi và chỉ khi các số đôi một nguyên tố cùng nhau**. Vì vậy đáp án là `max(2, độ dài lớn nhất của đoạn gồm các số đôi một nguyên tố cùng nhau)`.
5. Tránh các lỗi kinh điển: quên rằng **mọi đoạn dài 2** đều thỏa (đáp án luôn `>= 2`), chỉ kiểm tra `gcd == 1` (chưa đủ), dùng "các giá trị khác nhau" thay cho "nguyên tố cùng nhau", tính gcd/lcm của **cả mảng** thay vì mảng con, và co cửa sổ bằng `if` thay vì `while`.

## 2. Đề bài & ví dụ

Cho mảng số nguyên **dương** `nums`. Một mảng `arr` được gọi là **product equivalent** nếu:

```
prod(arr) == lcm(arr) * gcd(arr)
```

trong đó `prod(arr)` là **tích** mọi phần tử, `gcd(arr)` là **ước chung lớn nhất** của mọi phần tử và `lcm(arr)` là **bội chung nhỏ nhất** của mọi phần tử.

Trả về **độ dài của mảng con product equivalent dài nhất** của `nums`. (Mảng con là một đoạn **liên tiếp** các phần tử.)

**Ví dụ 1**

- Input: `nums = [1,2,1,2,1,1,1]`
- Output: `5`
- Giải thích: mảng con dài nhất là `[1,2,1,1,1]`: `prod = 2`, `gcd = 1`, `lcm = 2` ⇒ `2 == 2 * 1`.

**Ví dụ 2**

- Input: `nums = [2,3,4,5,6]`
- Output: `3`
- Giải thích: mảng con dài nhất là `[3,4,5]`.

**Ví dụ 3**

- Input: `nums = [1,2,3,1,4,5,1]`
- Output: `5`

**Ràng buộc:** `2 <= nums.length <= 100`; `1 <= nums[i] <= 10`.

In [1]:
import random
from itertools import product
from math import gcd, lcm, prod

examples = [
    ([1, 2, 1, 2, 1, 1, 1], 5),
    ([2, 3, 4, 5, 6], 3),
    ([1, 2, 3, 1, 4, 5, 1], 5),
]
for nums, expected in examples:
    print(nums, "-> kỳ vọng", expected)

[1, 2, 1, 2, 1, 1, 1] -> kỳ vọng 5
[2, 3, 4, 5, 6] -> kỳ vọng 3
[1, 2, 3, 1, 4, 5, 1] -> kỳ vọng 5


## 3. Kiến thức nền

### 3.1. Ước chung lớn nhất `math.gcd` và bội chung nhỏ nhất `math.lcm`

- `gcd(a, b)`: số lớn nhất chia hết cả `a` và `b`. `gcd(12, 18) = 6`.
- `lcm(a, b)`: số nhỏ nhất mà cả `a` và `b` đều chia hết. `lcm(4, 6) = 12`.
- Từ Python 3.9, cả hai nhận **nhiều đối số**: `gcd(a, b, c)`, `lcm(a, b, c)`. `gcd()` không đối số cho `0`, `lcm()` cho `1` (phần tử "trung tính").

In [2]:
print(gcd(12, 18), lcm(4, 6))              # 6 12
print(gcd(12, 18, 30), lcm(2, 3, 4))       # 6 12
print(gcd(), lcm())                         # 0 1  (phần tử trung tính)
print(gcd(0, 7), gcd(1, 9))                 # 7 1

6 12
6 12
0 1
7 1


### 3.2. Tích `math.prod` và vì sao Python không sợ tràn số

`prod(iterable)` nhân mọi phần tử (`prod([]) = 1`). Python có **số nguyên lớn tùy ý**, nên tích của `100` số tới `10` (tối đa `10^100`) vẫn chính xác, không tràn như `int` 32-bit của C++/Java (đó là lý do lời giải ở ngôn ngữ khác phải cắt sớm bằng `max_p`).

In [3]:
print(prod([2, 3, 4]))                 # 24
print(prod([]))                         # 1
print(prod([10] * 30))                  # 10**30, không tràn
print(prod([10] * 30) == 10 ** 30)     # True

24
1
1000000000000000000000000000000
True


### 3.3. Đẳng thức cho hai số: `gcd(a, b) * lcm(a, b) == a * b`

Với **hai** số nguyên dương luôn có `gcd(a, b) · lcm(a, b) = a · b`. Vậy **mọi mảng con dài 2** đều product equivalent. Với **ba số trở lên** đẳng thức `prod == lcm · gcd` **không còn luôn đúng**: đó là chỗ bài toán trở nên thú vị.

In [4]:
for a, b in [(4, 6), (3, 5), (2, 2), (1, 7), (8, 12)]:
    print(a, b, "| gcd*lcm =", gcd(a, b) * lcm(a, b), "| a*b =", a * b)
    assert gcd(a, b) * lcm(a, b) == a * b

arr = [2, 4, 3]                          # ba số: không còn đúng
print(prod(arr), gcd(*arr) * lcm(*arr))   # 24 vs 12

4 6 | gcd*lcm = 24 | a*b = 24
3 5 | gcd*lcm = 15 | a*b = 15
2 2 | gcd*lcm = 4 | a*b = 4
1 7 | gcd*lcm = 7 | a*b = 7
8 12 | gcd*lcm = 96 | a*b = 96
24 12


### 3.4. Số nguyên tố và "nguyên tố cùng nhau"

Hai số **nguyên tố cùng nhau** (coprime) nếu `gcd = 1`, tức **không có số nguyên tố nào chia hết cả hai**. Với các số `1..10` chỉ có bốn số nguyên tố liên quan: `2, 3, 5, 7`. Phân tích: `4 = 2²`, `6 = 2·3`, `8 = 2³`, `9 = 3²`, `10 = 2·5`, còn `1` **không có** thừa số nguyên tố (nguyên tố cùng nhau với mọi số).

In [5]:
print(gcd(4, 9) == 1, gcd(6, 10) == 1, gcd(1, 8) == 1)    # True False True
primes_of = {1: [], 2: [2], 3: [3], 4: [2], 5: [5], 6: [2, 3], 7: [7], 8: [2], 9: [3], 10: [2, 5]}
for x in range(1, 11):
    print(x, "->", primes_of[x])

True False True
1 -> []
2 -> [2]
3 -> [3]
4 -> [2]
5 -> [5]
6 -> [2, 3]
7 -> [7]
8 -> [2]
9 -> [3]
10 -> [2, 5]


### 3.5. Mặt nạ bit (bitmask) cho tập số nguyên tố

Gán mỗi số nguyên tố một bit: `2 → 1 (0b0001)`, `3 → 2 (0b0010)`, `5 → 4 (0b0100)`, `7 → 8 (0b1000)`. Mỗi số `x` có **mặt nạ** là OR các bit của thừa số nguyên tố của nó, ví dụ `6 → 0b0011 = 3`, `10 → 0b0101 = 5`, `1 → 0`. Khi đó:

- hai số **nguyên tố cùng nhau** ⇔ `mask[a] & mask[b] == 0` (không chung bit);
- gộp tập: `used |= mask[x]`; bỏ khỏi tập (khi các mặt nạ rời nhau): `used ^= mask[x]`.

In [6]:
MASK = {1: 0, 2: 1, 3: 2, 4: 1, 5: 4, 6: 3, 7: 8, 8: 1, 9: 2, 10: 5}
print(MASK[6] & MASK[10])                   # 1  (chung thừa số 2) -> không nguyên tố cùng nhau
print(MASK[4] & MASK[9], MASK[3] & MASK[5]) # 0 0 -> nguyên tố cùng nhau
used = 0
used |= MASK[3]; used |= MASK[4]            # thêm 3 và 4
print(bin(used))                             # 0b11
used ^= MASK[3]                              # bỏ 3 (mặt nạ rời nhau nên xor bỏ đúng)
print(bin(used))                             # 0b1
for a in range(1, 11):                       # kiểm tra MASK khớp với gcd
    for b in range(1, 11):
        assert (MASK[a] & MASK[b] == 0) == (gcd(a, b) == 1)
print("MASK khớp gcd cho mọi cặp 1..10")

1
0 0
0b11
0b1
MASK khớp gcd cho mọi cặp 1..10


### 3.6. Cửa sổ trượt (hai con trỏ `l`, `r`)

Để tìm đoạn dài nhất thỏa một điều kiện giữ được khi **thu nhỏ** đoạn, ta dùng hai con trỏ: `r` đi từ trái sang phải; mỗi lần thêm `nums[r]` mà điều kiện bị phá, ta **tăng `l`** (bỏ phần tử bên trái) cho đến khi điều kiện đúng lại. Độ dài `r - l + 1` luôn hợp lệ. Mỗi con trỏ chỉ đi một chiều nên tổng `O(n)`.

In [7]:
# Ví dụ nhỏ: đoạn dài nhất không có hai số bằng nhau
nums = [1, 2, 1, 3, 2]
seen = set()
l = 0
best = 0
for r, x in enumerate(nums):
    while x in seen:              # vi phạm: bỏ phần tử bên trái đến khi hết trùng
        seen.remove(nums[l])
        l += 1
    seen.add(x)
    best = max(best, r - l + 1)
    print(f"r={r}, x={x}, l={l}, cửa sổ={nums[l:r + 1]}")
print("dài nhất:", best)

r=0, x=1, l=0, cửa sổ=[1]
r=1, x=2, l=0, cửa sổ=[1, 2]
r=2, x=1, l=1, cửa sổ=[2, 1]
r=3, x=3, l=1, cửa sổ=[2, 1, 3]
r=4, x=2, l=2, cửa sổ=[1, 3, 2]
dài nhất: 3


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Kiểm tra từng mảng con nhỏ của `nums = [2,3,4,5,6]`:

| mảng con | `prod` | `gcd` | `lcm` | `gcd·lcm` | thỏa? |
|---|---|---|---|---|---|
| `[2,3]` | 6 | 1 | 6 | 6 | có (dài 2: luôn có) |
| `[3,4]` | 12 | 1 | 12 | 12 | có |
| `[2,3,4]` | 24 | 1 | 12 | 12 | **không** (`2` và `4` chung thừa số `2`) |
| `[3,4,5]` | 60 | 1 | 60 | 60 | **có** (đôi một nguyên tố cùng nhau) |
| `[4,5,6]` | 120 | 1 | 60 | 60 | không (`4` và `6` chung `2`) |
| `[3,4,5,6]` | 360 | 1 | 60 | 60 | không |

Quan sát: các mảng con dài `>= 3` thỏa **đúng khi các số đôi một nguyên tố cùng nhau** (`[3,4,5]`). Khi có hai số **chung một thừa số nguyên tố** (như `2` và `4`), tích bị "đếm thừa" thừa số đó mà `gcd · lcm` thì không.

In [8]:
for arr in ([2, 3], [3, 4], [2, 3, 4], [3, 4, 5], [4, 5, 6], [3, 4, 5, 6]):
    P, G, L = prod(arr), gcd(*arr), lcm(*arr)
    print(f"{str(arr):<14} prod={P:<4} gcd={G} lcm={L:<3} gcd*lcm={G * L:<4} thỏa={P == G * L}")

[2, 3]         prod=6    gcd=1 lcm=6   gcd*lcm=6    thỏa=True
[3, 4]         prod=12   gcd=1 lcm=12  gcd*lcm=12   thỏa=True
[2, 3, 4]      prod=24   gcd=1 lcm=12  gcd*lcm=12   thỏa=False
[3, 4, 5]      prod=60   gcd=1 lcm=60  gcd*lcm=60   thỏa=True
[4, 5, 6]      prod=120  gcd=1 lcm=60  gcd*lcm=60   thỏa=False
[3, 4, 5, 6]   prod=360  gcd=1 lcm=60  gcd*lcm=60   thỏa=False


## 5. Cách 1 — Brute Force / baseline: tính trực tiếp cho mọi mảng con

**Ý tưởng.** Duyệt mọi cặp `(i, j)`, lấy `sub = nums[i:j+1]`, tính `prod(sub)`, `gcd(*sub)`, `lcm(*sub)` rồi so sánh theo **đúng định nghĩa** của đề. Cập nhật đáp án bằng độ dài lớn nhất.

In [9]:
def max_length_brute(nums: list[int]) -> int:
    n = len(nums)
    best = 0
    for i in range(n):
        for j in range(i, n):
            sub = nums[i:j + 1]
            if prod(sub) == gcd(*sub) * lcm(*sub):       # đúng định nghĩa của đề
                best = max(best, j - i + 1)
    return best


for nums, expected in examples:
    print(max_length_brute(nums), "(kỳ vọng", expected, ")")

5 (kỳ vọng 5 )
3 (kỳ vọng 3 )
5 (kỳ vọng 5 )


**Độ phức tạp (Cách 1).** Gọi `n = len(nums)`. Có `O(n²)` mảng con; mỗi mảng con tính `prod`, `gcd`, `lcm` tốn `O(n)` ⇒ **Time `O(n³)`**, **Space `O(n)`** cho `sub`. Với `n <= 100` là cỡ `10^5` phép tính — chấp nhận được.

**Nút thắt:** với cùng điểm bắt đầu `i`, mảng con `[i..j+1]` chỉ hơn `[i..j]` **một phần tử**; tính lại từ đầu là lãng phí. Ta có thể **cập nhật dần** `prod`, `gcd`, `lcm` khi kéo dài `j`.

## 6. Ý tưởng thuật toán / bước đột phá

**Bước 1 — cập nhật dần.** Với `i` cố định, khi thêm `nums[j]`: `P *= nums[j]`, `G = gcd(G, nums[j])`, `L = lcm(L, nums[j])`. Mỗi bước `O(1)` ⇒ cả bài `O(n²)`.

**Bước 2 — hiểu cấu trúc bằng số mũ nguyên tố (đột phá toán học).** Xét một số nguyên tố `p` và các số mũ `e_1, ..., e_k` của `p` trong `k` phần tử của mảng. Khi đó số mũ của `p` trong `prod` là `Σ e_i`, trong `lcm` là `max e_i`, trong `gcd` là `min e_i`. Điều kiện của đề với `p` là:

```
Σ e_i = max e_i + min e_i
```

- Nếu `k = 2`: luôn đúng (hai số hạng chính là max và min).
- Nếu `k >= 3`: các số mũ **ở giữa** (không phải max, min) phải cộng bằng `0`, nên đều bằng `0`; mà `min <= các số ở giữa = 0` nên `min = 0` luôn. Tức là **ít nhất `k - 1` phần tử không chia hết cho `p`**, hay **tối đa một** phần tử chia hết cho `p`.

Điều đó đúng với **mọi** nguyên tố `p` ⇔ **không có nguyên tố nào chia hết hai phần tử** ⇔ các phần tử **đôi một nguyên tố cùng nhau**.

**Kết luận:**

- đoạn dài `1`: thỏa ⇔ phần tử bằng `1` (vì `x = x · x` ⇔ `x = 1`);
- đoạn dài `2`: **luôn** thỏa;
- đoạn dài `>= 3`: thỏa ⇔ **đôi một nguyên tố cùng nhau**.

Vì `n >= 2` nên có ít nhất một đoạn dài `2` ⇒ **đáp án `>= 2`**, và đáp án `= max(2, độ dài lớn nhất của đoạn đôi một nguyên tố cùng nhau)`.

Bước 3 (cửa sổ trượt) dùng tính chất: **bỏ bớt phần tử khỏi đoạn đôi một nguyên tố cùng nhau thì vẫn đôi một nguyên tố cùng nhau**, nên dùng hai con trỏ được.

In [10]:
# Kiểm chứng "định lý" bằng vét cạn mọi mảng độ dài 1..4 với giá trị 1..10
def is_product_equivalent(arr):
    return prod(arr) == gcd(*arr) * lcm(*arr)


def pairwise_coprime(arr):
    return all(gcd(arr[a], arr[b]) == 1 for a in range(len(arr)) for b in range(a + 1, len(arr)))


def predicted(arr):
    if len(arr) == 1:
        return arr[0] == 1
    if len(arr) == 2:
        return True
    return pairwise_coprime(arr)


checked = 0
for length in range(1, 5):
    for arr in product(range(1, 11), repeat=length):
        assert is_product_equivalent(arr) == predicted(arr), arr
        checked += 1
print("định lý đúng trên", checked, "mảng")

định lý đúng trên 11110 mảng


## 7. Các cách tiếp theo

### Cách 2 — Cập nhật dần `prod`, `gcd`, `lcm` (O(n²))

**Giải thích từng đoạn.**

1. Với mỗi `i`, đặt `P = 1`, `G = 0`, `L = 1` (phần tử trung tính của `prod`, `gcd`, `lcm`).
2. Với `j` từ `i` đến `n - 1`: `P *= nums[j]`, `G = gcd(G, nums[j])`, `L = lcm(L, nums[j])`.
3. Nếu `P == G * L` thì cập nhật `best = max(best, j - i + 1)`.

In [11]:
def max_length_incremental(nums: list[int]) -> int:
    n = len(nums)
    best = 0
    for i in range(n):
        P, G, L = 1, 0, 1                      # trung tính: prod=1, gcd=0, lcm=1
        for j in range(i, n):
            P *= nums[j]
            G = gcd(G, nums[j])
            L = lcm(L, nums[j])
            if P == G * L:
                best = max(best, j - i + 1)
    return best


for nums, expected in examples:
    print(max_length_incremental(nums), "(kỳ vọng", expected, ")")

5 (kỳ vọng 5 )
3 (kỳ vọng 3 )
5 (kỳ vọng 5 )


In [12]:
# Code nộp LeetCode cho Cách 2 (đổi tên lớp thành Solution khi nộp)
from math import gcd, lcm


class SolutionIncremental:
    def maxLength(self, nums: list[int]) -> int:
        n = len(nums)
        best = 0
        for i in range(n):
            P, G, L = 1, 0, 1
            for j in range(i, n):
                P *= nums[j]
                G = gcd(G, nums[j])
                L = lcm(L, nums[j])
                if P == G * L:
                    best = max(best, j - i + 1)
        return best


print(SolutionIncremental().maxLength([2, 3, 4, 5, 6]))      # 3

3


**Độ phức tạp Cách 2:** **Time `O(n²)`** (cộng thêm chi phí nhân/gcd/lcm với số lớn nhưng Python xử lý nhanh), **Space `O(1)`**. Lưu ý trong các ngôn ngữ có số nguyên cố định, tích có thể tràn: lời giải gốc cắt sớm khi `P > lcm(cả mảng) * max(cả mảng)` vì từ đó trở đi không thể thỏa lại.

### Cách 3 — Dùng tính chất "đôi một nguyên tố cùng nhau"

**Giải thích.** Theo mục 6, ta chỉ cần tìm đoạn dài nhất mà các phần tử **đôi một nguyên tố cùng nhau**. Với mỗi `i`, kéo dài `j` **chừng nào `nums[j]` nguyên tố cùng nhau với mọi phần tử đã có trong đoạn** (`gcd(nums[j], x) == 1` với mọi `x` trong `nums[i:j]`); vi phạm thì `break` vì mở rộng tiếp chỉ có thể xấu hơn. Cuối cùng trả về `max(2, longest)`.

In [13]:
def max_length_coprime(nums: list[int]) -> int:
    n = len(nums)
    longest = 0
    for i in range(n):
        for j in range(i, n):
            # nums[j] phải nguyên tố cùng nhau với mọi phần tử trước nó trong đoạn [i, j)
            if all(gcd(nums[j], nums[k]) == 1 for k in range(i, j)):
                longest = max(longest, j - i + 1)
            else:
                break                          # đoạn dài hơn chứa cặp vi phạm, dừng
    return max(2, longest)                     # mọi đoạn dài 2 đều thỏa


for nums, expected in examples:
    print(max_length_coprime(nums), "(kỳ vọng", expected, ")")

5 (kỳ vọng 5 )
3 (kỳ vọng 3 )
5 (kỳ vọng 5 )


In [14]:
# Code nộp LeetCode cho Cách 3
from math import gcd


class SolutionCoprime:
    def maxLength(self, nums: list[int]) -> int:
        n = len(nums)
        longest = 0
        for i in range(n):
            for j in range(i, n):
                if all(gcd(nums[j], nums[k]) == 1 for k in range(i, j)):
                    longest = max(longest, j - i + 1)
                else:
                    break
        return max(2, longest)


print(SolutionCoprime().maxLength([1, 2, 3, 1, 4, 5, 1]))    # 5

5


**Độ phức tạp Cách 3:** **Time `O(n³)`** trường hợp xấu (đoạn toàn số `1`: mỗi lần thêm phải so với mọi phần tử trước), nhưng với `n <= 100` là cỡ `10^6` phép `gcd` — nhanh. **Space `O(1)`**.

### Cách 4 — Cửa sổ trượt với mặt nạ bit (O(n))

**Giải thích từng đoạn.**

1. `MASK[x]` là mặt nạ các số nguyên tố của `x` (mục 3.5); `used` là OR các mặt nạ của cửa sổ `nums[l..r-1]` (các mặt nạ **rời nhau** vì cửa sổ luôn đôi một nguyên tố cùng nhau).
2. Với `x = nums[r]`, **trong khi** `used & MASK[x] != 0` (còn số nguyên tố trùng): bỏ `nums[l]` bằng `used ^= MASK[nums[l]]` rồi `l += 1`.
3. Sau đó `used |= MASK[x]` và cập nhật `best = max(best, r - l + 1)`.
4. Trả về `max(2, best)`.

In [15]:
MASK = {1: 0, 2: 1, 3: 2, 4: 1, 5: 4, 6: 3, 7: 8, 8: 1, 9: 2, 10: 5}


def max_length_window_mask(nums: list[int]) -> int:
    used = 0                                   # tập số nguyên tố đang có trong cửa sổ
    l = 0
    best = 0
    for r, x in enumerate(nums):
        m = MASK[x]
        while used & m:                        # x chung số nguyên tố với cửa sổ: thu nhỏ bên trái
            used ^= MASK[nums[l]]
            l += 1
        used |= m
        best = max(best, r - l + 1)
    return max(2, best)


for nums, expected in examples:
    print(max_length_window_mask(nums), "(kỳ vọng", expected, ")")

5 (kỳ vọng 5 )
3 (kỳ vọng 3 )
5 (kỳ vọng 5 )


In [16]:
# Code nộp LeetCode cho Cách 4
class SolutionWindowMask:
    def maxLength(self, nums: list[int]) -> int:
        mask = {1: 0, 2: 1, 3: 2, 4: 1, 5: 4, 6: 3, 7: 8, 8: 1, 9: 2, 10: 5}
        used = l = best = 0
        for r, x in enumerate(nums):
            m = mask[x]
            while used & m:
                used ^= mask[nums[l]]
                l += 1
            used |= m
            best = max(best, r - l + 1)
        return max(2, best)


print(SolutionWindowMask().maxLength([1, 2, 1, 2, 1, 1, 1]))   # 5

5


**Độ phức tạp Cách 4:** mỗi con trỏ chỉ đi tới `n - 1` ⇒ **Time `O(n)`**, **Space `O(1)`** (bảng `MASK` có `10` mục).

### Cách 5 — Cửa sổ trượt với phân tích thừa số nguyên tố tổng quát

**Giải thích.** Cách 4 gắn với giá trị `<= 10`. Nếu giá trị lớn hơn (ví dụ `<= 10^6`) ta **phân tích thừa số nguyên tố** `x` thành tập `ps` bằng chia thử, rồi dùng `dict count[p]` đếm số phần tử trong cửa sổ chia hết cho `p`. Điều kiện xung đột: tồn tại `p` trong `ps` mà `count[p] > 0`. Khi bỏ `nums[l]` thì giảm `count` của các nguyên tố của nó. Thuật toán giống hệt Cách 4 nhưng không phụ thuộc bảng cố định.

In [17]:
def prime_set(x: int) -> set[int]:
    ps = set()
    d = 2
    while d * d <= x:                         # chia thử
        if x % d == 0:
            ps.add(d)
            while x % d == 0:
                x //= d
        d += 1
    if x > 1:                                 # phần còn lại là một nguyên tố
        ps.add(x)
    return ps


print(prime_set(1), prime_set(10), prime_set(12), prime_set(49))   # set() {2, 5} {2, 3} {7}


def max_length_window_factor(nums: list[int]) -> int:
    count = {}                                # count[p] = số phần tử trong cửa sổ chia hết cho p
    l = 0
    best = 0
    for r, x in enumerate(nums):
        ps = prime_set(x)
        while any(count.get(p, 0) > 0 for p in ps):      # xung đột nguyên tố
            for p in prime_set(nums[l]):
                count[p] -= 1
            l += 1
        for p in ps:
            count[p] = count.get(p, 0) + 1
        best = max(best, r - l + 1)
    return max(2, best)


for nums, expected in examples:
    print(max_length_window_factor(nums), "(kỳ vọng", expected, ")")

set() {2, 5} {2, 3} {7}
5 (kỳ vọng 5 )
3 (kỳ vọng 3 )
5 (kỳ vọng 5 )


**Độ phức tạp Cách 5:** với `V` là giá trị lớn nhất, mỗi phân tích `O(√V)`; mỗi phần tử được thêm/bỏ một lần ⇒ **Time `O(n · √V)`**, **Space `O(số nguyên tố trong cửa sổ)`**. Ở bài này `V = 10` nên rất nhanh.

## 8. TRACE từng bước

Ta trace Cách 4 trên `nums = [1,2,3,1,4,5,1]` (đáp án `5`). `MASK`: `1→0`, `2→1`, `3→2`, `4→1`, `5→4`.

| `r` | `x` | `m` | `used` trước | xung đột (`used & m`)? | `l` sau thu nhỏ | `used` sau | cửa sổ | `best` |
|---|---|---|---|---|---|---|---|---|
| 0 | 1 | 0 | `0000` | không | 0 | `0000` | `[1]` | 1 |
| 1 | 2 | 1 | `0000` | không | 0 | `0001` | `[1,2]` | 2 |
| 2 | 3 | 2 | `0001` | không | 0 | `0011` | `[1,2,3]` | 3 |
| 3 | 1 | 0 | `0011` | không | 0 | `0011` | `[1,2,3,1]` | 4 |
| 4 | 4 | 1 | `0011` | **có** (bit 2) | 2 (bỏ `1`, bỏ `2`) | `0011` | `[3,1,4]` | 4 |
| 5 | 5 | 4 | `0011` | không | 2 | `0111` | `[3,1,4,5]` | 4 |
| 6 | 1 | 0 | `0111` | không | 2 | `0111` | `[3,1,4,5,1]` | **5** |

Ở bước `r = 4`: `4` có mặt nạ `0001` trùng bit của `2` đang ở cửa sổ, nên ta bỏ `nums[0] = 1` (mặt nạ `0`, không đổi `used`) rồi bỏ `nums[1] = 2` (xóa bit `0001`); lúc đó hết xung đột và thêm `4`.

In [18]:
def trace_max_length(nums):
    mask = {1: 0, 2: 1, 3: 2, 4: 1, 5: 4, 6: 3, 7: 8, 8: 1, 9: 2, 10: 5}
    used = 0
    l = 0
    best = 0
    print("nums =", nums)
    for r, x in enumerate(nums):
        m = mask[x]
        print(f"r={r}, x={x}, mask={m:04b}, used trước={used:04b}, xung đột={bool(used & m)}")
        while used & m:
            print(f"    bỏ nums[{l}]={nums[l]} (mask {mask[nums[l]]:04b}), l -> {l + 1}")
            used ^= mask[nums[l]]
            l += 1
        used |= m
        best = max(best, r - l + 1)
        print(f"    cửa sổ={nums[l:r + 1]}, used={used:04b}, best={best}")
    print("kết quả:", max(2, best))
    return max(2, best)


trace_max_length([1, 2, 3, 1, 4, 5, 1])

nums = [1, 2, 3, 1, 4, 5, 1]
r=0, x=1, mask=0000, used trước=0000, xung đột=False
    cửa sổ=[1], used=0000, best=1
r=1, x=2, mask=0001, used trước=0000, xung đột=False
    cửa sổ=[1, 2], used=0001, best=2
r=2, x=3, mask=0010, used trước=0001, xung đột=False
    cửa sổ=[1, 2, 3], used=0011, best=3
r=3, x=1, mask=0000, used trước=0011, xung đột=False
    cửa sổ=[1, 2, 3, 1], used=0011, best=4
r=4, x=4, mask=0001, used trước=0011, xung đột=True
    bỏ nums[0]=1 (mask 0000), l -> 1
    bỏ nums[1]=2 (mask 0001), l -> 2
    cửa sổ=[3, 1, 4], used=0011, best=4
r=5, x=5, mask=0100, used trước=0011, xung đột=False
    cửa sổ=[3, 1, 4, 5], used=0111, best=4
r=6, x=1, mask=0000, used trước=0111, xung đột=False
    cửa sổ=[3, 1, 4, 5, 1], used=0111, best=5
kết quả: 5


5

## 9. Code nộp LeetCode

Ta chọn **Cách 4** (cửa sổ trượt với mặt nạ bit): `O(n)`, ngắn gọn, và có chứng minh toán học đi kèm ở mục 6 và 10. (Nếu muốn an toàn tuyệt đối về mặt "bám đề", Cách 2 `O(n²)` cũng nộp được.)

In [19]:
class Solution:
    def maxLength(self, nums: list[int]) -> int:
        # mặt nạ bit của tập số nguyên tố chia hết mỗi số 1..10 (bit0=2, bit1=3, bit2=5, bit3=7)
        mask = {1: 0, 2: 1, 3: 2, 4: 1, 5: 4, 6: 3, 7: 8, 8: 1, 9: 2, 10: 5}
        used = 0                                # các nguyên tố đang có trong cửa sổ
        l = 0
        best = 0
        for r, x in enumerate(nums):
            m = mask[x]
            while used & m:                      # x chung nguyên tố với cửa sổ: thu nhỏ bên trái
                used ^= mask[nums[l]]
                l += 1
            used |= m
            best = max(best, r - l + 1)
        return max(2, best)                      # mọi đoạn dài 2 đều thỏa


for nums, expected in examples:
    print(Solution().maxLength(nums), "(kỳ vọng", expected, ")")

5 (kỳ vọng 5 )
3 (kỳ vọng 3 )
5 (kỳ vọng 5 )


## 10. Vì sao đúng

- **Đặc trưng toán học.** Với mỗi số nguyên tố `p`, số mũ của `p` trong `prod`, `lcm`, `gcd` lần lượt là `Σ e_i`, `max e_i`, `min e_i`. `prod = lcm · gcd` ⇔ với **mọi** `p`: `Σ e_i = max e_i + min e_i`. Với `k = 2` luôn đúng. Với `k >= 3`: `Σ e_i − max − min = Σ` các số mũ **ở giữa** `= 0` ⇒ chúng đều bằng `0` ⇒ `min = 0` ⇒ chỉ **tối đa một** phần tử chia hết cho `p`. Đúng với mọi `p` ⇔ đôi một nguyên tố cùng nhau. (Mục 6 cũng đã kiểm chứng bằng vét cạn.)
- **Đáp án `>= 2`.** Mọi đoạn dài `2` thỏa (`gcd·lcm = a·b`) và `n >= 2`, nên `max(2, ...)` bù cho trường hợp đoạn đôi một nguyên tố cùng nhau dài nhất chỉ có `1` phần tử (ví dụ `[3, 3]`, `[4, 6]`).
- **Cửa sổ trượt đúng.** Cửa sổ `nums[l..r]` luôn **đôi một nguyên tố cùng nhau** (invariant). Khi thêm `x`: nếu `x` trùng nguyên tố với cửa sổ thì phải bỏ từ trái cho đến khi hết trùng (bỏ bớt phần tử không phá tính chất đôi một nguyên tố cùng nhau). Với mỗi `r`, `l` là **vị trí nhỏ nhất** sao cho `nums[l..r]` đôi một nguyên tố cùng nhau — vì vị trí hợp lệ nhỏ nhất **không giảm** khi `r` tăng (nếu `nums[l..r]` hợp lệ thì `nums[l..r-1]` cũng hợp lệ), nên hai con trỏ chỉ đi tới.
- **Mặt nạ rời nhau ⇒ `xor` bỏ đúng.** Trong cửa sổ các mặt nạ không chung bit, nên `used` là hợp rời; `used ^= mask[nums[l]]` xóa đúng các bit của phần tử bỏ đi mà không ảnh hưởng phần tử khác.
- **Số `1`.** Mặt nạ `0` nên `1` không bao giờ xung đột — đúng, vì `gcd(1, ·) = 1`.

## 11. Time/Space Complexity

Gọi `n = len(nums)`.

- **Time `O(n)`** cho cửa sổ trượt: `r` đi `n` bước; `l` chỉ tăng, tối đa `n` bước tổng cộng; mỗi bước `O(1)` (phép bit, tra bảng).
- **Space `O(1)`**: chỉ có bảng `mask` kích thước cố định (`10` mục), vài biến nguyên.

So với Cách 2 (`O(n²)`) hay Cách 1 (`O(n³)`): với `n <= 100` mọi cách đều chạy tức thì; cách `O(n)` có ý nghĩa nếu `n` lớn (`10^5`).

## 12. Bộ test

### 12.1. Ví dụ của đề và gom các cách vào một bảng

In [20]:
impls = {
    "brute": max_length_brute,
    "incremental": max_length_incremental,
    "coprime": max_length_coprime,
    "window_mask": max_length_window_mask,
    "window_factor": max_length_window_factor,
    "Solution": lambda nums: Solution().maxLength(nums),
}

for nums, expected in examples:
    for name, fn in impls.items():
        assert fn(nums) == expected, (name, nums)
print("PASS các ví dụ của đề với", len(impls), "cách")

PASS các ví dụ của đề với 6 cách


### 12.2. Edge case

- `n = 2`: luôn trả `2` (mọi đoạn dài 2 thỏa), dù hai số bằng nhau hoặc chung nhiều thừa số (`[2,2]`, `[4,6]`, `[10,10]`).
- Toàn số `1`: cả mảng thỏa (`prod = 1`, `gcd = 1`, `lcm = 1`).
- Toàn một giá trị `> 1` (`[2,2,2,2]`): chỉ đoạn dài `2` thỏa.
- Cửa sổ nguyên tố cùng nhau cực đại: `[2,3,5,7]` (bốn nguyên tố), và có thêm số `1` ở hai đầu.
- Cặp vi phạm nằm giữa/hai đầu: `[2,3,4]` (`2` và `4` ở hai đầu) ⇒ `2`.
- `gcd = 1` nhưng không đôi một nguyên tố cùng nhau: `[6, 10, 3]` ⇒ `2`.

In [21]:
cases = [
    ([2, 2], 2),
    ([4, 6], 2),
    ([10, 10], 2),
    ([1, 1], 2),
    ([1] * 100, 100),
    ([2, 2, 2, 2], 2),
    ([2, 3, 5, 7], 4),
    ([1, 2, 3, 5, 7, 1], 6),
    ([2, 3, 4], 2),
    ([6, 10, 3], 2),
    ([4, 9, 5], 3),
    ([8, 9, 5, 7, 10], 4),         # [8,9,5,7] đôi một nguyên tố cùng nhau; 10 chung thừa số 5 với 5 nên không thêm được
    ([2, 3, 2, 3, 2, 3], 2),
    ([1, 2, 1, 4, 1, 3, 1], 5),    # [1,4,1,3,1] dài 5; [1,2,1] chỉ 3
]
for nums, expected in cases:
    for name, fn in impls.items():
        assert fn(nums) == expected, (name, nums, fn(nums), expected)
print("PASS", len(cases), "edge case x", len(impls), "cách")

PASS 14 edge case x 6 cách


### 12.3. Kiểm chứng định lý và phản ví dụ của các cách làm sai

Ta (1) kiểm tra lại định lý "đôi một nguyên tố cùng nhau" trên mọi mảng độ dài `1..4`, (2) dựng đầu vào làm lộ lỗi của các cách sai ở mục 13.

In [22]:
def wrong_gcd_only(nums):
    """SAI: chỉ cần gcd của cửa sổ bằng 1."""
    n = len(nums)
    best = 0
    for i in range(n):
        for j in range(i, n):
            if gcd(*nums[i:j + 1]) == 1:
                best = max(best, j - i + 1)
    return best


def wrong_distinct_values(nums):
    """SAI: cửa sổ gồm các giá trị khác nhau."""
    seen, l, best = set(), 0, 0
    for r, x in enumerate(nums):
        while x in seen:
            seen.remove(nums[l])
            l += 1
        seen.add(x)
        best = max(best, r - l + 1)
    return max(2, best)


def wrong_no_floor_2(nums):
    """SAI: quên đáp án luôn >= 2."""
    return max_length_window_mask_no_floor(nums)


def max_length_window_mask_no_floor(nums):
    used = l = best = 0
    for r, x in enumerate(nums):
        m = MASK[x]
        while used & m:
            used ^= MASK[nums[l]]
            l += 1
        used |= m
        best = max(best, r - l + 1)
    return best                                  # thiếu max(2, ...)


tests = [([6, 10, 3], wrong_gcd_only), ([2, 4, 3], wrong_distinct_values), ([2, 2], wrong_no_floor_2), ([4, 6], wrong_no_floor_2)]
for nums, wrong in tests:
    print(f"{nums} | {wrong.__name__}: {wrong(nums)} | đúng: {Solution().maxLength(nums)}")
    assert wrong(nums) != Solution().maxLength(nums)

[6, 10, 3] | wrong_gcd_only: 3 | đúng: 2
[2, 4, 3] | wrong_distinct_values: 3 | đúng: 2
[2, 2] | wrong_no_floor_2: 1 | đúng: 2
[4, 6] | wrong_no_floor_2: 1 | đúng: 2


### 12.4. Đối chiếu ngẫu nhiên (có seed) đúng ràng buộc

- **Mảng ngắn** (`n` từ `2` đến `25`): so **mọi** cách, kể cả cách brute theo đúng định nghĩa.
- **Mảng dài** (`n` tới `100`): bỏ brute `O(n³)` (chậm), so các cách còn lại.

Giá trị `1..10`. Để có nhiều mảng "khó" ta thay đổi **bảng giá trị sử dụng**: chỉ `{1,2}`, `{1,3,5,7}` (nguyên tố cùng nhau nhiều), hoặc cả `1..10`.

In [23]:
rng = random.Random(3411)
pools = [[1, 2], [1, 3, 5, 7], [1, 2, 3, 5, 7], [2, 3, 4, 5, 6], list(range(1, 11))]
big_answers = 0
for trial in range(2500):
    n = rng.randint(2, 25)
    pool = rng.choice(pools)
    nums = [rng.choice(pool) for _ in range(n)]
    assert 2 <= len(nums) <= 100 and all(1 <= x <= 10 for x in nums)
    ref = max_length_brute(nums)
    for name, fn in impls.items():
        assert fn(nums) == ref, (name, nums)
print("PASS 2500 mảng ngắn (đối chiếu cả brute)")

fast = {k: v for k, v in impls.items() if k != "brute"}
for trial in range(400):
    n = rng.randint(26, 100)
    pool = rng.choice(pools)
    nums = [rng.choice(pool) for _ in range(n)]
    ref = max_length_incremental(nums)
    big_answers += ref > 10
    for name, fn in fast.items():
        assert fn(nums) == ref, (name, nums)
print("PASS 400 mảng dài; số mảng có đáp án > 10:", big_answers)

PASS 2500 mảng ngắn (đối chiếu cả brute)


PASS 400 mảng dài; số mảng có đáp án > 10: 11


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Chỉ kiểm tra `gcd == 1`

In [24]:
def sai_chi_gcd_1(nums):
    best = 0
    for i in range(len(nums)):
        for j in range(i, len(nums)):
            if gcd(*nums[i:j + 1]) == 1:               # SAI: gcd = 1 chưa đủ
                best = max(best, j - i + 1)
    return best


print("sai:", sai_chi_gcd_1([6, 10, 3]), "| đúng:", Solution().maxLength([6, 10, 3]))
sub = [6, 10, 3]
print("prod =", prod(sub), "| gcd*lcm =", gcd(*sub) * lcm(*sub))

sai: 3 | đúng: 2
prod = 180 | gcd*lcm = 30


**Vì sao sai:** `gcd(6, 10, 3) = 1` nhưng `6` và `10` chung thừa số `2`, `6` và `3` chung thừa số `3`. Tích `180` còn `gcd · lcm = 1 · 30 = 30`. Điều kiện đúng là **đôi một** nguyên tố cùng nhau, không phải gcd cả nhóm bằng `1`. **Cách sửa:** kiểm tra từng cặp (hoặc dùng mặt nạ nguyên tố).

### Lỗi 2: Dùng "các giá trị khác nhau" thay cho "nguyên tố cùng nhau"

In [25]:
def sai_khac_nhau(nums):
    seen, l, best = set(), 0, 0
    for r, x in enumerate(nums):
        while x in seen:                               # SAI: chỉ loại trùng giá trị
            seen.remove(nums[l])
            l += 1
        seen.add(x)
        best = max(best, r - l + 1)
    return max(2, best)


print("sai:", sai_khac_nhau([2, 4, 3]), "| đúng:", Solution().maxLength([2, 4, 3]))

sai: 3 | đúng: 2


**Vì sao sai:** `2` và `4` là hai giá trị khác nhau nhưng chung nguyên tố `2`; mảng `[2, 4, 3]` có `prod = 24`, `gcd · lcm = 1 · 12 = 12`. Điều kiện xung đột phải xét theo **nguyên tố chung**, không theo giá trị bằng nhau. **Cách sửa:** dùng mặt nạ bit hoặc `count[p]`.

### Lỗi 3: Quên rằng đáp án luôn `>= 2`

In [26]:
def sai_quen_san_2(nums):
    used = l = best = 0
    for r, x in enumerate(nums):
        m = MASK[x]
        while used & m:
            used ^= MASK[nums[l]]
            l += 1
        used |= m
        best = max(best, r - l + 1)
    return best                                         # SAI: thiếu max(2, best)


print("sai:", sai_quen_san_2([2, 2]), "| đúng:", Solution().maxLength([2, 2]))
print("sai:", sai_quen_san_2([4, 6]), "| đúng:", Solution().maxLength([4, 6]))

sai: 1 | đúng: 2
sai: 1 | đúng: 2


**Vì sao sai:** với hai số bất kỳ ta có `gcd · lcm = a · b` nên **mọi đoạn dài 2** đều thỏa — dù hai số bằng nhau hay chung nhiều thừa số. Cửa sổ đôi một nguyên tố cùng nhau chỉ phản ánh các đoạn dài `>= 3` (và `1`, `2` khi tình cờ nguyên tố cùng nhau). **Cách sửa:** `return max(2, best)` (vì `n >= 2`).

### Lỗi 4: Tính gcd/lcm của cả mảng thay vì của mảng con

In [27]:
def sai_ca_mang(nums):
    G, L, P = gcd(*nums), lcm(*nums), prod(nums)        # SAI: tính trên cả mảng
    return len(nums) if P == G * L else 0


for nums in ([2, 3, 4, 5, 6], [1, 2, 1, 2, 1, 1, 1]):
    print(nums, "| sai:", sai_ca_mang(nums), "| đúng:", Solution().maxLength(nums))

[2, 3, 4, 5, 6] | sai: 0 | đúng: 3
[1, 2, 1, 2, 1, 1, 1] | sai: 0 | đúng: 5


**Vì sao sai:** đề hỏi mảng con **dài nhất** thỏa điều kiện, nên điều kiện phải được tính cho **từng mảng con**, không phải cả mảng. **Cách sửa:** hai vòng lặp (hoặc cửa sổ trượt) và cập nhật `best`.

### Lỗi 5: Co cửa sổ bằng `if` thay vì `while`

In [28]:
def sai_if_thay_while(nums):
    used = l = best = 0
    for r, x in enumerate(nums):
        m = MASK[x]
        if used & m:                                    # SAI: chỉ bỏ MỘT phần tử, có thể chưa hết xung đột
            used ^= MASK[nums[l]]
            l += 1
        used |= m
        best = max(best, r - l + 1)
    return max(2, best)


print("sai:", sai_if_thay_while([2, 3, 6, 5, 7]), "| đúng:", Solution().maxLength([2, 3, 6, 5, 7]))

sai: 4 | đúng: 3


**Vì sao sai:** thêm `6 = 2·3` vào cửa sổ `[2, 3]` xung đột với **cả hai** phần tử, nên phải bỏ **liên tiếp nhiều** phần tử bên trái cho đến khi hết xung đột. `if` chỉ bỏ một phần tử rồi vẫn thêm `x` — cửa sổ giữ lại một cặp xung đột và độ dài bị tính thừa. **Cách sửa:** `while used & m:`.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "**`prod == lcm * gcd`**" ⇒ nghĩ ngay tới **số mũ nguyên tố**: `Σ e = max + min`. Hai số: luôn đúng; từ ba số: **đôi một nguyên tố cùng nhau**.
- Điều kiện "đôi một không chung ước nguyên tố" + "đoạn liên tiếp dài nhất" ⇒ **cửa sổ trượt** với tập nguyên tố đang dùng (mặt nạ bit khi chỉ có ít nguyên tố).
- Giá trị nhỏ (`<= 10`) ⇒ chỉ có `4` nguyên tố (`2, 3, 5, 7`) ⇒ **mặt nạ 4 bit**; giá trị lớn hơn ⇒ phân tích thừa số + `dict` đếm.
- `n <= 100` ⇒ cứ viết `O(n²)` brute để làm **đáp án chuẩn**, rồi tối ưu bằng toán và đối chiếu bằng test ngẫu nhiên.
- Trong Python không lo tràn số khi tính `prod`; ở C++/Java phải cắt sớm hoặc dùng tính chất toán học.
- Đừng quên **đáp án sàn** (`>= 2`) khi bài có tính chất đúng hiển nhiên cho độ dài nhỏ.
- `math.gcd(*arr)`, `math.lcm(*arr)` nhận nhiều đối số (Python 3.9+).

## 15. Biến thể của bài

1. **Đếm số mảng con** product equivalent thay vì độ dài lớn nhất.
2. **Giá trị lớn hơn** (`<= 10^6`): dùng Cách 5 (phân tích thừa số nguyên tố, `dict` đếm).
3. **Điều kiện khác:** `prod == lcm` (các số đôi một nguyên tố cùng nhau, kể cả độ dài 2) hoặc `lcm == gcd * k`.
4. **Đoạn dài nhất đôi một nguyên tố cùng nhau** (đúng chính xác, không có đáp án sàn 2) — là bước trung gian của bài này.
5. Bài liên quan: **Longest Substring Without Repeating Characters (3)**, **Greatest Common Divisor of Strings (1071)**, **Number of Subarrays With GCD Equal to K (2447)**, **Number of Subarrays With LCM Equal to K (2470)**, **Split the Array to Make Coprime Products (2584)**.

In [29]:
def longest_pairwise_coprime(nums):
    """Đoạn dài nhất đôi một nguyên tố cùng nhau (không có đáp án sàn 2)."""
    used = l = best = 0
    for r, x in enumerate(nums):
        m = MASK[x]
        while used & m:
            used ^= MASK[nums[l]]
            l += 1
        used |= m
        best = max(best, r - l + 1)
    return best


def count_product_equivalent(nums):
    n = len(nums)
    return sum(prod(nums[i:j + 1]) == gcd(*nums[i:j + 1]) * lcm(*nums[i:j + 1]) for i in range(n) for j in range(i, n))


def count_prod_equals_lcm(nums):
    """Đếm mảng con có prod == lcm."""
    n = len(nums)
    return sum(prod(nums[i:j + 1]) == lcm(*nums[i:j + 1]) for i in range(n) for j in range(i, n))


print(longest_pairwise_coprime([2, 2]))                       # 1
print(count_product_equivalent([2, 3, 4]))                    # [2],[3],[4] không; [2,3],[3,4] có; [2,3,4] không -> 2
print(count_prod_equals_lcm([2, 3, 4]))                       # [2],[3],[4],[2,3],[3,4] -> 5
assert longest_pairwise_coprime([2, 2]) == 1
assert longest_pairwise_coprime([1, 2, 3, 1, 4, 5, 1]) == 5
assert count_product_equivalent([2, 3, 4]) == 2
assert count_prod_equals_lcm([2, 3, 4]) == 5

1
2
5


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Brute theo định nghĩa | `prod`, `gcd`, `lcm` của mọi mảng con | `O(n³)` | `O(n)` | đáp án chuẩn, `n` rất nhỏ |
| 2. Cập nhật dần | `P *= x`, `G = gcd(G, x)`, `L = lcm(L, x)` | `O(n²)` | `O(1)` | bám sát đề, đủ nhanh với `n <= 100` |
| 3. Đôi một nguyên tố cùng nhau | kiểm `gcd(nums[j], nums[k]) == 1`, `break` khi vi phạm | `O(n³)` xấu | `O(1)` | kiểm chứng tính chất toán học |
| 4. Cửa sổ trượt + mặt nạ bit | `used & mask` | `O(n)` | `O(1)` | **nên nộp** (giá trị `<= 10`) |
| 5. Cửa sổ trượt + thừa số nguyên tố | `dict` đếm nguyên tố | `O(n·√V)` | `O(π(V))` | giá trị lớn, tổng quát |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: `gcd(12, 18)`, `lcm(4, 6)`, `prod([2, 3, 4])`, `Solution().maxLength([5, 5])`, `Solution().maxLength([2, 3, 5])`.

**Bài 2 (tính tay).** Với `nums = [2, 3, 6, 5]`: với mỗi mảng con dài `>= 3` ghi `prod`, `gcd`, `lcm`, `gcd·lcm` và kết luận có thỏa không; từ đó suy ra đáp án.

**Bài 3 (điền TODO).** Hoàn thành `max_length_todo(nums)` theo Cách 4 (cửa sổ trượt + mặt nạ bit).

In [30]:
def max_length_todo(nums):
    mask = {1: 0, 2: 1, 3: 2, 4: 1, 5: 4, 6: 3, 7: 8, 8: 1, 9: 2, 10: 5}
    used = 0
    l = 0
    best = 0
    # TODO 1: duyệt r, x trong enumerate(nums); m = mask[x]
    #   TODO 2: while used & m: bỏ nums[l] bằng used ^= mask[nums[l]]; l += 1
    #   TODO 3: used |= m; cập nhật best = max(best, r - l + 1)
    # TODO 4: trả về max(2, best)
    pass

# KHÔNG gọi hàm TODO ở đây để khi Run All không bị lỗi (TODO chưa điền nên kết quả chưa đúng).

**Bài 4 (tự viết lại).** Viết `max_length_incremental2(nums)` theo Cách 2 nhưng thêm **tối ưu cắt sớm**: nếu `P > lcm(*nums) * max(nums)` thì `break` (tích đã vượt ngưỡng mà không đoạn nào dài hơn có thể thỏa lại). Kiểm tra kết quả khớp Cách 2.

**Bài 5 (biến thể / mini challenge).** (a) Viết `count_subarrays_coprime(nums)` đếm số mảng con **dài `>= 1`** có các phần tử đôi một nguyên tố cùng nhau (dùng cửa sổ trượt: mỗi `r` cộng `r - l + 1`). (b) Viết `max_length_large(nums)` cho giá trị tới `10^6` bằng phân tích thừa số nguyên tố (dùng `prime_set` ở Cách 5).

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: `6`, `12`, `24`, `2` (đoạn dài 2 luôn thỏa), `3` (`2, 3, 5` đôi một nguyên tố cùng nhau). Bài 3: theo mục 9, chỉ cần điền vòng lặp.
- **Mức 2 (vừa):** Bài 2: `[2,3,6]`: `prod 36`, `gcd 1`, `lcm 6` ⇒ `6 != 36` không thỏa; `[3,6,5]`: `prod 90`, `gcd 1`, `lcm 30` ⇒ không; `[2,3,6,5]`: `prod 180`, `gcd 1`, `lcm 30` ⇒ không. Đáp án `2` (một đoạn dài 2 bất kỳ). Bài 4: `ceiling = lcm(*nums) * max(nums)`; trong vòng `j`: `if P > ceiling: break`.
- **Mức 3 (sâu):** Bài 5(a): giữ `used`, `l` như Cách 4; mỗi `r` sau khi co cửa sổ cộng `r - l + 1` vào tổng (mọi đoạn kết thúc ở `r` bắt đầu từ `l..r` đều hợp lệ). 5(b): thay `used & m` bằng kiểm tra `any(count[p] > 0 for p in ps)`, như Cách 5; ràng buộc mới chỉ ảnh hưởng chi phí phân tích thừa số.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [31]:
# Bài 1
assert gcd(12, 18) == 6
assert lcm(4, 6) == 12
assert prod([2, 3, 4]) == 24
assert Solution().maxLength([5, 5]) == 2
assert Solution().maxLength([2, 3, 5]) == 3

# Bài 2
nums2 = [2, 3, 6, 5]
for sub in ([2, 3, 6], [3, 6, 5], [2, 3, 6, 5]):
    assert prod(sub) != gcd(*sub) * lcm(*sub)
assert Solution().maxLength(nums2) == 2


# Bài 3
def max_length_todo_ans(nums):
    mask = {1: 0, 2: 1, 3: 2, 4: 1, 5: 4, 6: 3, 7: 8, 8: 1, 9: 2, 10: 5}
    used = 0
    l = 0
    best = 0
    for r, x in enumerate(nums):
        m = mask[x]
        while used & m:
            used ^= mask[nums[l]]
            l += 1
        used |= m
        best = max(best, r - l + 1)
    return max(2, best)


for nums, expected in examples:
    assert max_length_todo_ans(nums) == expected


# Bài 4
def max_length_incremental2(nums):
    ceiling = lcm(*nums) * max(nums)
    n = len(nums)
    best = 0
    for i in range(n):
        P, G, L = 1, 0, 1
        for j in range(i, n):
            P *= nums[j]
            if P > ceiling:                          # tích đã quá lớn: các đoạn dài hơn không thể thỏa
                break
            G = gcd(G, nums[j])
            L = lcm(L, nums[j])
            if P == G * L:
                best = max(best, j - i + 1)
    return best


for nums, expected in examples:
    assert max_length_incremental2(nums) == expected
rng = random.Random(34110)
for _ in range(500):
    arr = [rng.randint(1, 10) for _ in range(rng.randint(2, 40))]
    assert max_length_incremental2(arr) == max_length_incremental(arr), arr


# Bài 5
def count_subarrays_coprime(nums):
    mask = {1: 0, 2: 1, 3: 2, 4: 1, 5: 4, 6: 3, 7: 8, 8: 1, 9: 2, 10: 5}
    used = l = total = 0
    for r, x in enumerate(nums):
        m = mask[x]
        while used & m:
            used ^= mask[nums[l]]
            l += 1
        used |= m
        total += r - l + 1
    return total


def max_length_large(nums):
    return max_length_window_factor(nums)


# [1,2,3]: đoạn 1: 3 đoạn; dài 2: 2 đoạn; dài 3: 1 đoạn -> 6 (đều đôi một nguyên tố cùng nhau)
assert count_subarrays_coprime([1, 2, 3]) == 6
assert count_subarrays_coprime([2, 4]) == 2                  # chỉ hai đoạn dài 1
assert max_length_large([999983, 2, 3, 999979, 5, 4]) == 5       # [999983,2,3,999979,5]; thêm 4 chung 2 với 2
assert max_length_large([2, 3, 5, 7, 11, 13]) == 6
print("PASS đáp án bài tập")

PASS đáp án bài tập


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (đoạn dài nhất đôi một nguyên tố cùng nhau bằng cửa sổ trượt + mặt nạ bit; đáp án sàn `2`)

```python
class Solution:
    def maxLength(self, nums: List[int]) -> int:
        mask = {1: 0, 2: 1, 3: 2, 4: 1, 5: 4, 6: 3, 7: 8, 8: 1, 9: 2, 10: 5}
        used = l = best = 0
        for r, x in enumerate(nums):
            m = mask[x]
            while used & m:
                used ^= mask[nums[l]]
                l += 1
            used |= m
            best = max(best, r - l + 1)
        return max(2, best)
```

**Checklist**

- [ ] Nhớ: `prod == lcm * gcd` ⇔ (độ dài 2) hoặc **đôi một nguyên tố cùng nhau**?
- [ ] Đáp án sàn `2` (mọi đoạn dài 2 đều thỏa, và `n >= 2`)?
- [ ] Mặt nạ bit khớp `gcd` (đã kiểm với mọi cặp `1..10`)?
- [ ] Co cửa sổ bằng `while` (không phải `if`) và bỏ phần tử bên trái bằng `^=`?
- [ ] Đã thử `[2,2]` (`2`), `[6,10,3]` (`2`), `[1]*100` (`100`), `[2,3,5,7]` (`4`) và ba ví dụ của đề (`5`, `3`, `5`)?
- [ ] Skeleton đúng `class Solution` / `def maxLength(self, nums: List[int]) -> int`?